1. Setup & Bibliotek
Här laddar vi verktygen som krävs för att bygga en klassisk ML-modell.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Felsökning
print(df.columns)

2. Data-laddning & "Feature Engineering"
Detta är den viktigaste cellen. En Random Forest kan inte läsa text som Gemma kan; den vill ha siffror. Vi måste omvandla JSON-datan till en tabell.

In [2]:
import pandas as pd
import numpy as np

# 1. Ladda data
raw_df = pd.read_json("network_audit_log.jsonl", lines=True)

# DIAGNOS: Kolla om filen faktiskt innehåller det vi tror
if 'metrics' not in raw_df.columns:
    print(f"❌ FEL: Kolumnen 'metrics' saknas! Hittade bara: {list(raw_df.columns)}")
    # Om kolumnen heter något annat (t.ex. 'state'), döp om den här:
    if 'state' in raw_df.columns:
        raw_df = raw_df.rename(columns={'state': 'metrics'})
        print("🔄 Döp om 'state' till 'metrics'...")

# Rensa rader där metrics saknas
df = raw_df.dropna(subset=['metrics']).copy()

def extract_features_final(row):
    try:
        metrics = row['metrics']
        
        # Om Gemma sparat ner metrics som en STRÄNG (händer ibland med Ollama/JSON)
        # så konverterar vi den till en dictionary här
        if isinstance(metrics, str):
            metrics = json.loads(metrics)
            
        srv = metrics.get("10.0.0.1", {})
        iot = metrics.get("192.168.1.100", {})
        
        srv_traffic = srv.get('traffic', 0)
        iot_traffic = iot.get('traffic', 0)
        
        is_scanning = 0
        for ip in metrics:
            device = metrics[ip]
            if isinstance(device, dict): # Säkerställ att det är en dict
                notes = str(device.get('notes', '')).lower()
                if 'scanning' in notes or 'port' in notes:
                    is_scanning = 1
                    break
        
        total = srv_traffic + iot_traffic
        load_ratio = srv_traffic / total if total > 0 else 0
        
        return pd.Series({
            'server_vol': float(srv_traffic),
            'iot_vol': float(iot_traffic),
            'behavior_flag': int(is_scanning),
            'load_ratio': float(load_ratio)
        })
    except Exception as e:
        return pd.Series({'server_vol': np.nan, 'iot_vol': np.nan, 'behavior_flag': np.nan, 'load_ratio': np.nan})

# 2. Kör appliceringen
X = df.apply(extract_features_final, axis=1)

# 3. Hantera resultatet och synka med y (besluten)
# Vi kollar om 'correct' finns, annars kollar vi efter 'decision'
label_col = 'correct' if 'correct' in df.columns else 'decision'

if label_col in df.columns:
    # Om 'decision' är text (t.ex. 'isolate'), gör om till 1/0
    if df[label_col].dtype == object:
        y_values = df[label_col].apply(lambda x: 1 if x == 'isolate' else 0)
    else:
        y_values = df[label_col].astype(int)
    
    # Ta bara rader som fungerade i både X och y
    valid_mask = X['server_vol'].notna()
    X = X[valid_mask].copy()
    y = y_values[valid_mask].copy()
else:
    print("❌ FEL: Hittade ingen kolumn för 'correct' eller 'decision' att använda som facit!")

if len(X) > 0:
    print(f"✅ Success! Extraherade {len(X)} rader.")
    print(X.head())
else:
    print("⚠️ VARNING: Inga rader extraherades. Kontrollera JSONL-filens format.")

❌ FEL: Kolumnen 'metrics' saknas! Hittade bara: ['timestamp', 'model', 'latency', 'decision', 'correct']


KeyError: ['metrics']

3. Träning av Random Forest
Nu lär vi studenten att se sambandet mellan avg_traffic och om det är en attack.

In [ ]:
# Dela upp i träning och test (80% träna, 20% testa)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Skapa och träna modellen
student_model = RandomForestClassifier(n_estimators=100)
student_model.fit(X_train, y_train)

print("✅ Student-modellen är färdigtränad!")

4. Utvärdering & Jämförelse
Här mäter vi hur bra studenten blev jämfört med läraren.

In [ ]:
y_pred = student_model.predict(X_test)
print(classification_report(y_test, y_pred))

# Visualisera vad modellen tyckte var viktigast (Feature Importance)
importances = pd.Series(student_model.feature_importances_, index=X.columns)
importances.plot(kind='barh', title="Vilka faktorer styr beslutet?")
plt.show()